# Notebook 04: Reproducible reporting

**Introduction to AI-Assisted Research Methods and Data Analysis** · Day 4

NexusHealth · Adamu Onu

This notebook rebuilds every reported number, table and figure from the analysis set, writes them to files, and renders the mini research report. If someone changes the data, one **Restart and Run All** should regenerate everything.

> **Teaching data only.** Every patient in this course is synthetic. The associations are simulated and are not clinical evidence. Never paste identifiable patient data, confidential manuscripts or restricted full text into a public AI tool.

### How to use this notebook

1. Run cells from top to bottom with **Shift + Enter**.
2. Before you run a cell, say out loud (or write) what you expect it to do. After it runs, check whether you were right. We call this *predict, run, inspect*.
3. Cells marked **Exercise** have gaps for you to fill. Cells marked **AI assist** suggest a prompt for GitHub Copilot Chat (or your own assistant).
4. If things get tangled, use **Restart** then **Run All** from the notebook toolbar.

# Part A: A reproducible workflow (block 1)

A result is reproducible when someone else can regenerate it from your files without asking you anything. Five habits get you most of the way:

1. **Run All from a fresh start.** Hidden state (a variable you made, then deleted the cell) is the most common reason notebooks break.
2. **Relative paths only**, built from the course folder.
3. **Assertions** that check the things you rely on.
4. **Outputs written by code**, never copied and pasted by hand.
5. **A record** of software versions and of how AI was used.

In [ ]:
suppressPackageStartupMessages({
  library(tidyverse)
})

# Find the course folder, wherever this notebook lives inside it.
ROOT <- normalizePath(getwd())
while (!dir.exists(file.path(ROOT, "data", "raw")) && dirname(ROOT) != ROOT) {
  ROOT <- dirname(ROOT)
}
stopifnot("Course folder not found. Open the whole course folder in VS Code." =
            dir.exists(file.path(ROOT, "data", "raw")))

DATA_RAW       <- file.path(ROOT, "data", "raw")
DATA_PROCESSED <- file.path(ROOT, "data", "processed")
DATA_REFERENCE <- file.path(ROOT, "data", "reference")
OUT_TABLES     <- file.path(ROOT, "outputs", "tables")
OUT_FIGURES    <- file.path(ROOT, "outputs", "figures")
for (folder in c(DATA_PROCESSED, OUT_TABLES, OUT_FIGURES)) dir.create(folder, recursive = TRUE, showWarnings = FALSE)

options(repr.plot.width = 7, repr.plot.height = 4.5, pillar.width = 120)
theme_set(theme_minimal(base_size = 12))
cat("Course folder found:", basename(ROOT), "\n")

In [ ]:
analysis_path <- file.path(DATA_PROCESSED, "analysis_set.csv")
if (!file.exists(analysis_path)) {
  analysis_path <- file.path(DATA_REFERENCE, "analysis_set_reference.csv")
  message("Your Day 2 file was not found, so the reference analysis set is used instead.")
}
df <- read_csv(analysis_path, show_col_types = FALSE) |>
  mutate(
    # Tell R which variables are categories, and which level is the reference.
    sex            = factor(sex, levels = c("Female", "Male")),
    smoking_status = factor(smoking_status, levels = c("Never", "Former", "Current")),
    clinic         = factor(clinic, levels = c("North", "East", "South", "West"))
  )
cat("Loaded", basename(analysis_path), ":", nrow(df), "rows,", ncol(df), "columns\n")

## A1. Check what we rely on

If any of these fail, stop and find out why before reporting anything.

In [ ]:
MODEL_VARS <- c("systolic_bp", "uncontrolled_bp", "adherence_10", "age", "sex", "bmi", "diabetes", "smoking_status")
stopifnot(
  "Duplicate patients in the analysis set" = !anyDuplicated(df$patient_id),
  "Missing values in model variables" = !anyNA(df[MODEL_VARS]),
  "Adherence outside 0 to 100" = all(between(df$adherence_10, 0, 10)),
  all(df$uncontrolled_bp %in% c(0, 1))
)
cat("Checks passed for", nrow(df), "patients.\n")

## A2. Record the software environment

In [ ]:
library(broom)
session <- tibble(
  item = c("date_run", "R", "tidyverse", "broom"),
  version = c(format(Sys.time(), "%Y-%m-%d %H:%M:%S"), paste(R.version$major, R.version$minor, sep = "."),
              as.character(packageVersion("tidyverse")), as.character(packageVersion("broom")))
)
dir.create(file.path(ROOT, "outputs", "logs"), recursive = TRUE, showWarnings = FALSE)
write_csv(session, file.path(ROOT, "outputs", "logs", "session_info.csv"))
session

## Exercise: Reproducibility audit

Open Notebook 02 and Notebook 03. For each: **Restart**, **Run All**, and note any error. Then answer: (1) Did both run without errors? (2) Did any number change from yesterday? (3) Find one place where a result depends on something outside the notebook (a file, a setting, a manual step) and say how it is protected.

**Your answer:** _Double-click here and write your answer._

# Part B: Tables, figures and the report (block 2)

## B1. Refit the models

This notebook fits the models again so that it can run on its own.

In [ ]:
COVARIATES <- "age + sex + bmi + factor(diabetes) + smoking_status"
models <- list(
  lin_crude = lm(systolic_bp ~ adherence_10, data = df),
  lin_adj   = lm(as.formula(paste("systolic_bp ~ adherence_10 +", COVARIATES)), data = df),
  log_crude = glm(uncontrolled_bp ~ adherence_10, data = df, family = binomial),
  log_adj   = glm(as.formula(paste("uncontrolled_bp ~ adherence_10 +", COVARIATES)), data = df, family = binomial)
)

# Wald intervals, so Python and R report identical numbers
effect <- function(model, term = "adherence_10", exponentiate = FALSE) {
  est <- coef(model)[[term]]
  ci <- confint.default(model)[term, ]
  p <- summary(model)$coefficients[term, 4]
  out <- c(est, ci)
  if (exponentiate) out <- exp(out)
  c(estimate = out[1], low = out[2], high = out[3], p = p)
}
fmt_p <- function(p) if (p < 0.001) "< 0.001" else sprintf("%.3f", p)

for (name in names(models)) {
  e <- effect(models[[name]], exponentiate = startsWith(name, "log"))
  cat(sprintf("%-10s %6.2f (%.2f to %.2f), p %s\n", name, e[1], e[2], e[3], fmt_p(e[4])))
}

## B2. Table 1: characteristics of the analysis set

Table 1 describes who was studied. We split it by outcome status, and use mean (SD) for roughly symmetric variables, median (IQR) for skewed ones, and n (%) for categories. We do **not** add p-values to Table 1: its job is description.

In [ ]:
mean_sd <- function(x) sprintf("%.1f (%.1f)", mean(x), sd(x))
median_iqr <- function(x) {
  x <- na.omit(x)
  sprintf("%.0f (%.0f to %.0f)", median(x), quantile(x, 0.25), quantile(x, 0.75))
}
n_pct <- function(x, value) sprintf("%d (%.1f)", sum(x == value), 100 * mean(x == value))

describe <- function(d) c(
  "Age, years, mean (SD)" = mean_sd(d$age),
  "Female, n (%)" = n_pct(d$sex, "Female"),
  "BMI, kg/m², mean (SD)" = mean_sd(d$bmi),
  "Diabetes, n (%)" = n_pct(d$diabetes, 1),
  "Smoking: never, n (%)" = n_pct(d$smoking_status, "Never"),
  "Smoking: former, n (%)" = n_pct(d$smoking_status, "Former"),
  "Smoking: current, n (%)" = n_pct(d$smoking_status, "Current"),
  "Medication adherence, %, mean (SD)" = mean_sd(d$medication_adherence),
  "Physical activity, min/week, median (IQR)†" = median_iqr(d$physical_activity_min),
  "Systolic BP, mmHg, mean (SD)" = mean_sd(d$systolic_bp),
  "Diastolic BP, mmHg, mean (SD)" = mean_sd(d$diastolic_bp)
)
controlled <- filter(df, uncontrolled_bp == 0)
uncontrolled <- filter(df, uncontrolled_bp == 1)
table1 <- tibble(
  Characteristic = names(describe(df)),
  !!sprintf("All (n = %d)", nrow(df)) := describe(df),
  !!sprintf("Controlled (n = %d)", nrow(controlled)) := describe(controlled),
  !!sprintf("Uncontrolled (n = %d)", nrow(uncontrolled)) := describe(uncontrolled)
)
table1_note <- sprintf("†Physical activity missing for %d patients. Uncontrolled BP: SBP ≥ 140 or DBP ≥ 90 mmHg (course definition).",
                       sum(is.na(df$physical_activity_min)))
table1

## B3. Table 2: the main results

A focused results table answers the research question directly: the adherence estimate, crude and adjusted, for both outcomes. The full adjusted models go in a supplementary table.

In [ ]:
est_ci <- function(model, exponentiate = FALSE, term = "adherence_10") {
  e <- effect(model, term, exponentiate)
  sprintf("%.2f (%.2f to %.2f)", e[1], e[2], e[3])
}

table2 <- tibble(
  Outcome = c("Systolic BP (mmHg)", "Uncontrolled BP"),
  `Effect measure` = c("Mean difference", "Odds ratio"),
  `Crude (95% CI)` = c(est_ci(models$lin_crude), est_ci(models$log_crude, TRUE)),
  `Adjusted (95% CI)` = c(est_ci(models$lin_adj), est_ci(models$log_adj, TRUE)),
  `Adjusted p` = c(fmt_p(effect(models$lin_adj)[4]), fmt_p(effect(models$log_adj)[4]))
)
table2_note <- sprintf(paste("Estimates per 10-point higher adherence score; n = %d. Adjusted for age, sex, BMI,",
                             "diabetes and smoking status. Wald 95%% confidence intervals."), nrow(df))

# Supplementary table: every term in both adjusted models
labels <- c("adherence_10" = "Adherence (per 10 points)", "age" = "Age (per year)", "sexMale" = "Male (vs female)",
            "bmi" = "BMI (per kg/m²)", "factor(diabetes)1" = "Diabetes (vs none)",
            "smoking_statusFormer" = "Former smoker (vs never)", "smoking_statusCurrent" = "Current smoker (vs never)")
tableS1 <- tibble(
  Term = labels,
  `SBP difference, mmHg (95% CI)` = map_chr(names(labels), ~ est_ci(models$lin_adj, FALSE, .x)),
  `Odds ratio (95% CI)` = map_chr(names(labels), ~ est_ci(models$log_adj, TRUE, .x))
)
print(table2)
tableS1

## B4. The main figure

A good results figure shows the estimate *and* its uncertainty on a scale readers understand. Here: the predicted probability of uncontrolled BP across the adherence range, for a reference patient (55 years old, female, BMI 28.5, no diabetes, never smoked), with a 95% confidence band. We save it at 300 dpi and write a text description for readers who cannot see it.

In [ ]:
reference <- tibble(adherence_10 = seq(1, 10, by = 0.1), age = 55,
                    sex = factor("Female", levels = c("Female", "Male")), bmi = 28.5, diabetes = 0,
                    smoking_status = factor("Never", levels = c("Never", "Former", "Current")))

# Predictions on the log-odds scale, then converted to probabilities
pred <- predict(models$log_adj, newdata = reference, type = "link", se.fit = TRUE)
reference <- reference |>
  mutate(prob = plogis(pred$fit),
         low = plogis(pred$fit - 1.96 * pred$se.fit),
         high = plogis(pred$fit + 1.96 * pred$se.fit))

fig1 <- ggplot(reference, aes(adherence_10 * 10, prob)) +
  geom_ribbon(aes(ymin = low, ymax = high), fill = "#A2B5CD", alpha = 0.5) +
  geom_line(colour = "#002366", linewidth = 1.2) +
  scale_y_continuous(labels = scales::percent, limits = c(0, 1)) +
  labs(x = "Medication adherence score (%)", y = "Probability of uncontrolled BP",
       title = "Predicted probability of uncontrolled BP by adherence",
       caption = "Shaded band: 95% confidence interval")
print(fig1)
ggsave(file.path(OUT_FIGURES, "figure1_predicted_probability.png"), fig1, width = 6.5, height = 4.2, dpi = 300)

at <- function(score) reference$prob[which.min(abs(reference$adherence_10 - score / 10))]
figure_alt <- sprintf(paste("Line chart. For a reference patient, the predicted probability of uncontrolled blood pressure",
                            "falls from %.0f%% at an adherence score of 20 to %.0f%% at 100.",
                            "The confidence band is widest at low adherence, where there are fewer patients."),
                      100 * at(2), 100 * at(10))
cat(figure_alt, "\n")

## B5. Write everything the report needs

The report never contains a hand-typed result. Tables are written as Markdown files, and the key numbers are written to `reports/_variables.yml`, which the report reads. Change the data, run this notebook again, and the report text updates itself.

In [ ]:
write_md_table <- function(table, path, note) {
  writeLines(c(knitr::kable(table, format = "pipe"), "", paste0("*", note, "*")), path, useBytes = TRUE)
}
write_md_table(table1, file.path(OUT_TABLES, "table1.md"), table1_note)
write_md_table(table2, file.path(OUT_TABLES, "table2.md"), table2_note)
write_md_table(tableS1, file.path(OUT_TABLES, "tableS1.md"), "Adjusted models; Wald 95% confidence intervals.")
write_csv(table1, file.path(OUT_TABLES, "table1.csv"))
write_csv(table2, file.path(OUT_TABLES, "table2.csv"))

ci_text <- function(e, unit = "") sprintf("%.2f%s (95%% CI %.2f to %.2f)", e[1], unit, e[2], e[3])
counterfactual <- map_dbl(c(40, 80), ~ mean(predict(models$log_adj, mutate(df, adherence_10 = .x / 10), type = "response")))
flow_path <- file.path(OUT_TABLES, "analysis_set_flow.csv")
variables <- c(
  n_analysis = nrow(df),
  n_clean = if (file.exists(flow_path)) read_csv(flow_path, show_col_types = FALSE)$n_remaining[1] else 640,
  pct_uncontrolled = sprintf("%.1f", 100 * mean(df$uncontrolled_bp)),
  lin_crude = ci_text(effect(models$lin_crude), " mmHg"),
  lin_adj = ci_text(effect(models$lin_adj), " mmHg"),
  log_crude = ci_text(effect(models$log_crude, exponentiate = TRUE)),
  log_adj = ci_text(effect(models$log_adj, exponentiate = TRUE)),
  prob_40 = sprintf("%.0f%%", 100 * counterfactual[1]),
  prob_80 = sprintf("%.0f%%", 100 * counterfactual[2]),
  figure_alt = figure_alt
)
lines <- sprintf('%s: "%s"', names(variables), variables)
writeLines(lines, file.path(ROOT, "reports", "_variables.yml"), useBytes = TRUE)
cat(lines, sep = "\n")

## Exercise 4A: Rewrite weak results prose

A first draft of the results said:

> *"Adherence was highly significant (p<0.001) for blood pressure, proving that taking medication lowers BP. Patients with good adherence had 19% less uncontrolled BP."*

**List what is wrong, then rewrite it** in two or three sentences using the numbers above.

**Your answer:** _Double-click here and write your answer._

## B6. Render the report

The report lives in `reports/mini_report.qmd`. It is a Quarto document: plain text with headings, plus instructions to include the tables and figure written above. The next cell asks Quarto to turn it into an HTML page and a Word document. If Quarto is not installed, use the instructions printed below, or render it from the VS Code command palette (**Quarto: Render**).

In [ ]:
if (nzchar(Sys.which("quarto"))) {
  old <- setwd(file.path(ROOT, "reports"))
  for (fmt in c("html", "docx")) {
    status <- system2("quarto", c("render", "mini_report.qmd", "--to", fmt), stdout = FALSE, stderr = FALSE)
    cat(fmt, if (status == 0) "rendered\n" else "FAILED (run quarto render in a terminal to see why)\n")
  }
  setwd(old)
  cat("Open reports/mini_report.html in your browser.\n")
} else {
  cat("Quarto was not found. Install it from https://quarto.org, then run in a terminal:\n",
      "  cd reports && quarto render mini_report.qmd\n")
}

# Part C: Structured discussion (block 3)

Use five labelled parts. Draft each in the discussion matrix (`templates/Discussion_Matrix_Template.docx`) before writing prose.

| Label | Part | Question it answers |
|---|---|---|
| **D** | Direct answer | What did we find, in one or two sentences, with the main estimate? |
| **L** | Literature | How does this compare with the verified sources in our literature matrix? |
| **F** | Framework | What does our conceptual or theoretical framework say this could mean? What did we *not* test? |
| **A** | Alternative explanations and limitations | Confounding, selection, measurement, missing data. For each: the likely direction and size of its effect. |
| **R** | Recommendation and next step | What should practice or the next study do, given the strength of evidence? |

**Claim and evidence audit.** For every sentence in your discussion that states a fact, write which evidence supports it: a cell in this notebook, a row in your literature matrix (with the exact passage), or *none*. Delete or soften any claim marked *none*.

## Exercise: Draft the D and A parts

Write the **D** part and one **A** limitation for the teaching case. For the limitation, name the problem, the likely direction of bias, and what would fix it.

**Your answer:** _Double-click here and write your answer._

# Part D: Capstone handover (block 4)

The handover package must let a reviewer regenerate every reported result. The cell below lists what should exist and flags anything missing.

In [ ]:
expected <- c(
  "data/raw/primary_care_raw.csv", "data/raw/data_dictionary.csv",
  "data/processed/analysis_set.csv", "outputs/tables/cleaning_log.csv",
  "outputs/tables/analysis_set_flow.csv", "outputs/tables/table1.md", "outputs/tables/table2.md",
  "outputs/figures/figure1_predicted_probability.png", "outputs/logs/session_info.csv",
  "outputs/logs/ai_use_log.csv", "reports/_variables.yml", "reports/mini_report.qmd",
  "reports/mini_report.html"
)
manifest <- tibble(file = expected, present = file.exists(file.path(ROOT, expected)))
write_csv(manifest, file.path(ROOT, "outputs", "handover_manifest.csv"))
manifest

Anything marked `False` needs attention before you submit. The capstone rubric is in the participant workbook.

## Key points

* Restart and Run All is the minimum test of a reproducible notebook.
* Write results to files from code; let the report read them.
* Table 1 describes; Table 2 answers the question; the figure shows size and uncertainty.
* Discussion: Direct answer, Literature, Framework, Alternatives and limitations, Recommendation.
* Every factual claim needs a source you have checked yourself.